# Naive Bayes (Manual)

In [1]:
import re
from collections import Counter, defaultdict

# Dataset from the prompt
data = [
    ("Free money now!!!", "SPAM"),
    ("Hi mom, how are you?", "HAM"),
    ("Lowest price for your meds", "SPAM"),
    ("Are we still on for dinner?", "HAM"),
    ("Win a free iPhone today", "SPAM"),
    ("Let's catch up tomorrow at the office", "HAM"),
    ("Meeting at 3 PM tomorrow", "HAM"),
    ("Get 50% off, limited time!", "SPAM"),
    ("Team meeting in the office", "HAM"),
    ("Click here for prizes!", "SPAM"),
    ("Can you send the report?", "HAM"),
]

def tokenize(text):
    # Keep only letters and digits, lowercase everything
    return re.findall(r"[a-z0-9]+", text.lower())

In [2]:
# 1) Bag of words (word frequencies per class and overall)
bow_overall = Counter()
bow_by_class = defaultdict(Counter)

for text, label in data:
    tokens = tokenize(text)
    bow_overall.update(tokens)
    bow_by_class[label].update(tokens)

vocab = sorted(bow_overall.keys())

print("Vocabulary size:", len(vocab))
print("Vocabulary (sorted):")
print(vocab)

print("\nBag of Words (overall):")
print(bow_overall)

print("\nBag of Words by class:")
for label in sorted(bow_by_class.keys()):
    print(label, bow_by_class[label])

Vocabulary size: 45
Vocabulary (sorted):
['3', '50', 'a', 'are', 'at', 'can', 'catch', 'click', 'dinner', 'for', 'free', 'get', 'here', 'hi', 'how', 'in', 'iphone', 'let', 'limited', 'lowest', 'meds', 'meeting', 'mom', 'money', 'now', 'off', 'office', 'on', 'pm', 'price', 'prizes', 'report', 's', 'send', 'still', 'team', 'the', 'time', 'today', 'tomorrow', 'up', 'we', 'win', 'you', 'your']

Bag of Words (overall):
Counter({'for': 3, 'the': 3, 'free': 2, 'are': 2, 'you': 2, 'tomorrow': 2, 'at': 2, 'office': 2, 'meeting': 2, 'money': 1, 'now': 1, 'hi': 1, 'mom': 1, 'how': 1, 'lowest': 1, 'price': 1, 'your': 1, 'meds': 1, 'we': 1, 'still': 1, 'on': 1, 'dinner': 1, 'win': 1, 'a': 1, 'iphone': 1, 'today': 1, 'let': 1, 's': 1, 'catch': 1, 'up': 1, '3': 1, 'pm': 1, 'get': 1, '50': 1, 'off': 1, 'limited': 1, 'time': 1, 'team': 1, 'in': 1, 'click': 1, 'here': 1, 'prizes': 1, 'can': 1, 'send': 1, 'report': 1})

Bag of Words by class:
HAM Counter({'the': 3, 'are': 2, 'you': 2, 'tomorrow': 2, 'at'

In [3]:
# 2) Class priors
class_counts = Counter(label for _, label in data)
total_docs = len(data)
priors = {label: class_counts[label] / total_docs for label in class_counts}

print("Class counts:", class_counts)
print("Class priors:", priors)

Class counts: Counter({'HAM': 6, 'SPAM': 5})
Class priors: {'SPAM': 0.45454545454545453, 'HAM': 0.5454545454545454}


In [4]:
# 3) Likelihoods with Laplace smoothing
alpha = 1.0
vocab_size = len(vocab)

likelihoods = {}
for label in class_counts:
    token_counts = bow_by_class[label]
    total_tokens = sum(token_counts.values())
    denom = total_tokens + alpha * vocab_size
    likelihoods[label] = {}
    for token in vocab:
        count = token_counts[token]
        likelihoods[label][token] = (count + alpha) / denom

for label in sorted(likelihoods.keys()):
    print(f"\nLikelihoods for {label} (Laplace smoothing):")
    for token in vocab:
        print(f"{token}: {likelihoods[label][token]:.6f}")


Likelihoods for HAM (Laplace smoothing):
3: 0.025316
50: 0.012658
a: 0.012658
are: 0.037975
at: 0.037975
can: 0.025316
catch: 0.025316
click: 0.012658
dinner: 0.025316
for: 0.025316
free: 0.012658
get: 0.012658
here: 0.012658
hi: 0.025316
how: 0.025316
in: 0.025316
iphone: 0.012658
let: 0.025316
limited: 0.012658
lowest: 0.012658
meds: 0.012658
meeting: 0.037975
mom: 0.025316
money: 0.012658
now: 0.012658
off: 0.012658
office: 0.037975
on: 0.025316
pm: 0.025316
price: 0.012658
prizes: 0.012658
report: 0.025316
s: 0.025316
send: 0.025316
still: 0.025316
team: 0.025316
the: 0.050633
time: 0.012658
today: 0.012658
tomorrow: 0.037975
up: 0.025316
we: 0.025316
win: 0.012658
you: 0.037975
your: 0.012658

Likelihoods for SPAM (Laplace smoothing):
3: 0.014925
50: 0.029851
a: 0.029851
are: 0.014925
at: 0.014925
can: 0.014925
catch: 0.014925
click: 0.029851
dinner: 0.014925
for: 0.044776
free: 0.044776
get: 0.029851
here: 0.029851
hi: 0.014925
how: 0.014925
in: 0.014925
iphone: 0.029851
let: 0.

In [5]:
# 4) Classify test sentences with Naive Bayes
test_sentences = [
    "Limited offer, click here!",
    "Meeting at 2 PM with the manager.",
]

def predict(text):
    tokens = tokenize(text)
    # Use log probabilities to avoid underflow
    scores = {}
    for label in class_counts:
        score = 0.0
        # Prior
        score += math.log(priors[label])
        # Likelihoods
        for token in tokens:
            if token in likelihoods[label]:
                score += math.log(likelihoods[label][token])
            else:
                # If token is unseen, apply Laplace smoothing implicitly
                denom = sum(bow_by_class[label].values()) + alpha * vocab_size
                score += math.log(alpha / denom)
        scores[label] = score
    # Return the label with the highest score
    return max(scores, key=scores.get), scores

import math

for sentence in test_sentences:
    label, scores = predict(sentence)
    print(f"Sentence: {sentence}")
    print(f"Predicted class: {label}")
    print(f"Scores: {scores}\n")

Sentence: Limited offer, click here!
Predicted class: SPAM
Scores: {'SPAM': -15.527786296248298, 'HAM': -18.083927213438404}

Sentence: Meeting at 2 PM with the manager.
Predicted class: HAM
Scores: {'SPAM': -30.221305696101027, 'HAM': -26.91560465182341}



Task 2

In [7]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
import numpy as np # Import numpy for np.exp

# Prepare data for scikit-learn
documents = [text for text, _ in data]
labels = [label for _, label in data]

vectorizer = CountVectorizer()
X_train = vectorizer.fit_transform(documents)
y_train = labels

nb_classifier = MultinomialNB(alpha=1.0)
nb_classifier.fit(X_train, y_train)


print("TASK 2a: SCIKIT-LEARN MODEL TRAINING")

print("\nModel trained successfully!")
print(f"Feature names (vocabulary): {vectorizer.get_feature_names_out()}")
print(f"\nTraining accuracy: {nb_classifier.score(X_train, y_train) * 100:.2f}%")
print(f"\nClass prior probabilities:")
for i, class_label in enumerate(nb_classifier.classes_):
    print(f"  P({class_label}) = {np.exp(nb_classifier.class_log_prior_[i]):.4f}")

TASK 2a: SCIKIT-LEARN MODEL TRAINING

Model trained successfully!
Feature names (vocabulary): ['50' 'are' 'at' 'can' 'catch' 'click' 'dinner' 'for' 'free' 'get' 'here'
 'hi' 'how' 'in' 'iphone' 'let' 'limited' 'lowest' 'meds' 'meeting' 'mom'
 'money' 'now' 'off' 'office' 'on' 'pm' 'price' 'prizes' 'report' 'send'
 'still' 'team' 'the' 'time' 'today' 'tomorrow' 'up' 'we' 'win' 'you'
 'your']

Training accuracy: 100.00%

Class prior probabilities:
  P(HAM) = 0.5455
  P(SPAM) = 0.4545


In [8]:
X_test = vectorizer.transform(test_sentences)

predictions = nb_classifier.predict(X_test)
probabilities = nb_classifier.predict_proba(X_test)

print("TASK 2b: CLASSIFY TEST SENTENCES (SCIKIT-LEARN)")

for i, test_sentence in enumerate(test_sentences):
    print(f"\nTest sentence {i+1}: '{test_sentence}'")
    print(f"Classification: {predictions[i]}")
    print(f"Probabilities:")
    for j, class_label in enumerate(nb_classifier.classes_):
        print(f"  P({class_label}|sentence) = {probabilities[i][j]:.6f}")

TASK 2b: CLASSIFY TEST SENTENCES (SCIKIT-LEARN)

Test sentence 1: 'Limited offer, click here!'
Classification: SPAM
Probabilities:
  P(HAM|sentence) = 0.084717
  P(SPAM|sentence) = 0.915283

Test sentence 2: 'Meeting at 2 PM with the manager.'
Classification: HAM
Probabilities:
  P(HAM|sentence) = 0.978443
  P(SPAM|sentence) = 0.021557


In [10]:
print("Manual v Scikit")

print("\nTest Sentence 1: 'Limited offer, click here!'")
print("  Manual Classification: SPAM")
print(f"  Scikit-Learn Classification: {predictions[0]}")

print("\nTest Sentence 2: 'Meeting at 2 PM with the manager.'")
print("  Manual Classification: HAM")
print(f"  Scikit-Learn Classification: {predictions[1]}")



Manual v Scikit

Test Sentence 1: 'Limited offer, click here!'
  Manual Classification: SPAM
  Scikit-Learn Classification: SPAM

Test Sentence 2: 'Meeting at 2 PM with the manager.'
  Manual Classification: HAM
  Scikit-Learn Classification: HAM


Conclusion:

Task 2 reimplemented the Naive Bayes spam classifier using scikit-learn's MultinomialNB with Laplace smoothing (α = 1.0) and CountVectorizer for feature extraction. The model achieved 100% training accuracy on the 11-message dataset and correctly classified both test sentences, "Limited offer, click here!" as SPAM (91.5% confidence) and "Meeting at 2 PM with the manager." as HAM (97.8% confidence). These results are in full agreement with the manual Naive Bayes implementation from Task 1, confirming that both approaches produce consistent predictions. This validates the correctness of the manual implementation while demonstrating how scikit-learn streamlines the same process with built-in vectorization, smoothing, and probability estimation.